<a href="https://colab.research.google.com/github/arsingla786/MLFromScratch/blob/main/SELECT_K_BEST_FEATURES(CHI2%2CIG).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [3]:
from sklearn.datasets import fetch_openml

In [7]:
dataset = fetch_openml('car')
df = dataset.data
df['target'] = dataset.target

/usr/local/lib/python3.13/dist-packages/sklearn/datasets/_openml.py:323: UserWarning: Multiple active versions of the dataset matching the name car exist. Versions may be fundamentally different, returning version 2. Available versions:
- version 2, status: active
  url: https://www.openml.org/search?type=data&id=991
- version 3, status: active
  url: https://www.openml.org/search?type=data&id=40975

  warn(warning_msg)


In [8]:
df

,buying,maint,doors,persons,lug_boot,safety,target
0,vhigh,vhigh,2,2,small,low,P
1,vhigh,vhigh,2,2,small,med,P
2,vhigh,vhigh,2,2,small,high,P
3,vhigh,vhigh,2,2,med,low,P
4,vhigh,vhigh,2,2,med,med,P
...,...,...,...,...,...,...,...
1723,low,low,5more,more,med,med,N
1724,low,low,5more,more,med,high,N
1725,low,low,5more,more,big,low,P
1726,low,low,5more,more,big,med,N


In [9]:
def chi_square_scratch(df,attribute,target):
    attr_values = df[attribute].unique()
    target_values = df[target].unique()
    chi2 = 0
    for attr_value in attr_values:
      for target_value in target_values:
        observed_freq = len(df[(df[attribute] == attr_value) & (df[target] == target_value)])
        attr_prob = len(df[df[attribute] == attr_value]) / len(df)
        target_prob = len(df[df[target] == target_value])/len(df)
        expected_freq = attr_prob * target_prob * len(df)
        chi2 += (observed_freq - expected_freq) ** 2 / expected_freq
    return chi2

In [10]:
chi_square_scratch(df,'buying','target')

76.5218545582182

In [14]:
from numpy._core.fromnumeric import sort
def selectkbest_chi2_scratch(df,k):
  attributes = df.columns[:-1]
  target = df.columns[-1]
  chi2_score = {}
  for attribute in attributes:
    chi2_score[attribute] = chi_square_scratch(df,attribute,target)
  sorted_chi2_score = sorted(chi2_score.items(),key=lambda x:x[1],reverse=True)
  selected_features = [f for f,_ in sorted_chi2_score[:k]]
  return selected_features,sorted_chi2_score

In [15]:
selectkbest_chi2_scratch(df,3)

(['safety', 'persons', 'buying'],
 [('safety', 396.34438878075235),
  ('persons', 370.2912281821373),
  ('buying', 76.5218545582182),
  ('maint', 64.17072657072656),
  ('lug_boot', 29.400095727368452),
  ('doors', 8.590650627014263)])

In [19]:
from scipy.stats import chi2_contingency
table = pd.crosstab(df['buying'],df['target'])
chi2_contingency(table)

Chi2ContingencyResult(statistic=np.float64(76.5218545582182), pvalue=np.float64(1.7095876176351842e-16), dof=3, expected_freq=array([[129.5, 302.5],
       [129.5, 302.5],
       [129.5, 302.5],
       [129.5, 302.5]]))

In [20]:
from scipy.stats.distributions import chi2
def SelectKbest_chi2_inbuilt(df,k):
  attributes  = df.columns[:-1]
  target = df.columns[-1]
  chi2_score = {}
  for attribute in attributes:
    table = pd.crosstab(df[attribute],df[target])
    chi2_score[attribute] = chi2_contingency(table)[0]
  sorted_chi2_score = sorted(chi2_score.items(),key=lambda x:x[1],reverse=True)
  selected_features = [f for f,_ in sorted_chi2_score[:k]]
  return selected_features

In [21]:
SelectKbest_chi2_inbuilt(df,3)

['safety', 'persons', 'buying']

In [25]:
from sklearn.preprocessing import LabelEncoder
le  = LabelEncoder()

for col in df.columns:
  df[col] = le.fit_transform(df[col])

from sklearn.feature_selection import SelectKBest,chi2
test = SelectKBest(chi2,k=3)
test.fit(df.iloc[:,:-1],df.iloc[:,-1])
test.scores_


array([  6.39795782,   8.75055362,   5.55984556, 177.86727081,
        18.53771977,  17.64446855])

In [26]:
df.columns[:-1][test.get_support()]

Index(['persons', 'lug_boot', 'safety'], dtype='object')

Information gain

In [27]:
def find_entropy(df):
  target = df.columns[-1]
  target_values = df[target].unique()
  entropy = 0
  for target_value in target_values:
    prob = len(df[df[target] == target_value]) / len(df)
    entropy += -prob * np.log(prob)
  return entropy


In [29]:
find_entropy(df)

np.float64(0.610668040691973)

In [30]:
def find_avg_info_entropy(df,attribute):
  attr_values = df[attribute].unique()
  target = df.columns[-1]
  target_values = df[target].unique()
  avg_info_entropy = 0
  for attribute_value in attr_values:
    entropy_subsample = 0
    for target_value in target_values:
      num = len(df[(df[attribute] == attribute_value) & (df[target] == target_value)])
      den = len(df[df[attribute] == attribute_value])
      prob = num / den
      entropy_subsample += -prob * np.log(prob + 1e-7)
    weight =  len(df[df[attribute] == attribute_value]) / len(df)
    avg_info_entropy += weight * entropy_subsample
  return avg_info_entropy

In [31]:
find_avg_info_entropy(df,'buying')

np.float64(0.5877249927287859)

In [32]:
def selectkbest_IG_scratch(df,k):
  attributes = df.columns[:-1]
  target = df.columns[-1]
  IG_score = {}
  for attribute in attributes:
    IG_score[attribute] = find_entropy(df) - find_avg_info_entropy(df,attribute)
  sorted_IG_score = sorted(IG_score.items(),key=lambda x:x[1],reverse=True)
  selected_features = [f for f,_ in sorted_IG_score[:k]]
  return selected_features,sorted_IG_score

In [33]:
selectkbest_IG_scratch(df,3)

(['safety', 'persons', 'buying'],
 [('safety', np.float64(0.15847366769495885)),
  ('persons', np.float64(0.15205711603543415)),
  ('buying', np.float64(0.022943047963187135)),
  ('maint', np.float64(0.01949543313852442)),
  ('lug_boot', np.float64(0.008726202068861544)),
  ('doors', np.float64(0.0025446115537233194))])

In [36]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

X = df.iloc[:, :-1]  # All columns except the last
y = df.iloc[:, -1]   # Last column (target)

selectkbest = SelectKBest(score_func=mutual_info_classif, k=3)

selectkbest.fit(X, y)

print(selectkbest.scores_)

[0.008523   0.02861344 0.         0.17379903 0.01144781 0.15347884]


In [37]:
df.columns[:-1][selectkbest.get_support()]

Index(['maint', 'persons', 'safety'], dtype='object')